# 03 - End-to-End C2 Evaluation

**Objective.** Reconcile the preserved 25-fictional-learner simulation when its external evidence is supplied, inspect the combined scorecard, and verify the evidence boundaries recorded in the manifest.

**Inputs.** `C2_RESEARCH_ROOT` optionally points to the external research workspace. Without it, the notebook uses the tracked manifest as historical evidence and clearly labels that path.

In [1]:
from pathlib import Path
import json, os
import pandas as pd
from evaluation_runner import evaluate_existing_simulation

HERE = Path.cwd().resolve()
assert (HERE / 'evaluation_runner.py').is_file(), 'Run the notebook from the evaluation folder.'
external = os.environ.get('C2_RESEARCH_ROOT')
RESEARCH_ROOT = Path(external).resolve() if external else None
live = evaluate_existing_simulation(RESEARCH_ROOT)
manifest = json.loads((HERE / 'results_manifest.json').read_text(encoding='utf-8'))
simulation = live if live.get('available') else manifest['results']['existing_simulation']
print('Evidence path:', 'external row-level reconciliation' if live.get('available') else 'tracked historical manifest (external rows absent)')

Evidence path: external row-level reconciliation


In [2]:
reported = simulation['reported']
recomputed = simulation['recomputed']
pd.DataFrame([
    {'measure': 'fictional learners', 'reported': reported['learners_executed'], 'recomputed': recomputed['learners']},
    {'measure': 'scored interactions', 'reported': reported['total_scored_questions'], 'recomputed': recomputed['scored_questions']},
    {'measure': 'ML-assisted recommendations', 'reported': reported['xgboost_recommendations'], 'recomputed': recomputed['trained_ml_assisted']},
    {'measure': 'cold-start fallbacks', 'reported': reported['deterministic_fallback_or_abstention'], 'recomputed': recomputed['rule_fallback']},
    {'measure': 'curriculum-valid selections', 'reported': reported['total_scored_questions'] - reported['grade_inappropriate_recommendations'], 'recomputed': recomputed['curriculum_valid']},
    {'measure': 'valid C4 exports', 'reported': reported['learners_executed'] if reported['all_c4_exports_valid'] else 0, 'recomputed': recomputed['c4_exports']},
])

,measure,reported,recomputed
0,fictional learners,25,25
1,scored interactions,489,489
2,ML-assisted recommendations,464,464
3,cold-start fallbacks,25,25
4,curriculum-valid selections,489,489
5,valid C4 exports,25,25


In [3]:
summary = pd.read_csv(HERE / 'results_summary.csv')
scorecard = [
    'Research winner ROC-AUC', 'Deployed transferable ROC-AUC',
    'Expanded XGBoost score change', 'Expanded candidate-ranking change',
    'Expanded final activity change', 'Expanded Curriculum Compliance Rate',
    'Expanded Tier-Policy Compliance', 'Expanded invariant stability',
    'Expanded session completion', 'C4 export validity',
    'Real-child recommendation utility'
]
summary[summary.metric_name.isin(scorecard)][['metric_name','value','numerator','denominator','evidence_category','interpretation','limitation']]

,metric_name,value,numerator,denominator,evidence_category,interpretation,limitation
0,Research winner ROC-AUC,0.750158,NaN,937043.0,A,Discrimination on held-out ASSISTments interac...,Not recommendation quality or local-child vali...
7,Deployed transferable ROC-AUC,0.712077,NaN,937043.0,A,Discrimination of TRANSFER_H2_GB_RT_OFF_v1,Source-domain result; not the 0.750158 researc...
13,Expanded final activity change,0.183333,22.0,120.0,C,Final activity changed after all constraints,Higher is not automatically better
15,Expanded XGBoost score change,0.925000,111.0,120.0,C,At least one common candidate-skill score changed,Source-domain probabilities
16,Expanded candidate-ranking change,0.625000,75.0,120.0,C,Candidate order around the policy target changed,Internal behavioral measure
19,Expanded Curriculum Compliance Rate,1.000000,280.0,280.0,C,Adherence to encoded curriculum rules,Not independent pedagogical validation
20,Expanded Tier-Policy Compliance,1.000000,280.0,280.0,C,Adherence to implemented tier rules,Not independent difficulty appropriateness
21,Expanded invariant stability,1.000000,20.0,20.0,C,Equivalent histories yielded identical observa...,Deterministic synthetic controls
27,Expanded session completion,1.000000,25.0,25.0,C,Completion within frozen evidence budget,Programmed synthetic responses
31,C4 export validity,1.000000,25.0,25.0,A/C,C4-compatible exports passed schema/consistenc...,Synthetic data only


In [4]:
display(pd.DataFrame(manifest['results']['recommendation']['baseline_session_comparison']).T)
display(manifest['verification'])
print('Tracked source commit:', manifest['results']['repository']['head'])
print('Educational utility status:', manifest['results']['educational_utility']['status'])

,completed,curriculum_compliance_rate,mean_skill_coverage,mean_valid_responses,recommendations,sessions,tier_policy_compliance_rate,trained_ml_usage_rate,valid_session_completion_rate
fixed_sequence,5.0,1.0,1.0,23.400000,351.0,15.0,1.0,0.000000,0.333333
hybrid,15.0,1.0,1.0,19.533333,293.0,15.0,1.0,0.948805,1.000000
rule_only,15.0,1.0,1.0,20.133333,302.0,15.0,1.0,0.000000,1.000000


{'status': 'Not supplied to this run'}

Tracked source commit: f449d5d9899a9444d63c72bcbe2e5e9c3791cb8c
Educational utility status: NOT_CAUSALLY_ESTIMABLE


## Interpretation and limitations

The 25 learners and their outcomes are fictional. Completion, fallback, curriculum selection, persistence, and C4 export results are software-integration evidence only. The activity-independent outcome generator makes learning-outcome comparisons invalid. Real-child recommendation utility remains not causally estimable.